# 5.7 訓練中斷怎麼接續？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：存檔像暫停遊戲：不只存角色外觀**

想接續 Adam，還要動量、步數與隨機狀態。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/checkpoint.svg")))

**先想一想：**Adam的動量不存，下一步還會一樣嗎？

只存模型權重能推論，但要精確接續還需要optimizer的動量、步數、tokenizer配置和隨機狀態。先確認重載forward相同，再測下一次更新。

**把直覺寫成數學：**checkpoint是狀態快照，不是文字檔改副檔名；模型架構與tokenizer必須匹配。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
import tempfile
from pathlib import Path
from tiny_perceptron.model import TinyLM, ModelConfig
from tiny_perceptron.training import save_checkpoint, load_checkpoint

model = TinyLM(ModelConfig(width=8))
with tempfile.TemporaryDirectory() as d:
    path = Path(d) / "state.pt"
    save_checkpoint(path, model, step=3)
    loaded, payload = load_checkpoint(path)
    x = torch.tensor([[1, 2, 3]])
    assert torch.equal(model(x)["logits"], loaded(x)["logits"])
    print(payload["step"], payload["tokenizer"])

**如何讀結果：**本節沒有正式訓練；下一步更新的一致性由tests/test_models.py檢查。只載可信來源checkpoint。

**只改一件事：**只用不同width的模型載入，觀察相容性錯誤。
